In [26]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI
import IPython

# If you get an error running this cell, then please head over to the troubleshooting notebook!

# Connecting to OpenAI

The next cell is where we load in the environment variables in your `.env` file and connect to OpenAI. 
The env file should look like this :

```
OPENAI_API_KEY=xxx
GOOGLE_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
HF_TOKEN=xxx
AZURE_OPENAI_API_KEY=xxx
```

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
openai = OpenAI()

# If this doesn't work, try Kernel menu >> Restart Kernel and Clear Outputs Of All Cells, then run the cells from the top of this notebook down.
# If it STILL doesn't work (horrors!) then please see the Troubleshooting notebook in this folder for full instructions

## And now let's build useful messages for GPT-4o-mini, using a function

# OpenAI Parameters

## 1. `model`
Specifies which model to use (e.g., `gpt-4`, `gpt-3.5-turbo`).  
- Different models have different capabilities, speed, and cost.  

## 2. `temperature`
Controls randomness in the output.  
- Range: `0.0 – 2.0`  
- Lower values → more deterministic, focused answers.  
- Higher values → more creative, varied responses.  

## 3. `max_tokens`
The maximum number of tokens (words + pieces of words) the model can generate.  
- Helps limit output length.  

## 4. `top_p`
Nucleus sampling parameter.  
- Range: `0.0 – 1.0`  
- The model considers only the most probable tokens that together have probability `p`.  
- Alternative to `temperature`.  

## 5. `frequency_penalty`
Controls how much to reduce the chance of repeating the same line/phrase.  
- Range: `-2.0 – 2.0`  
- Higher values → discourage repetition.  

## 6. `presence_penalty`
Controls how much to encourage the model to talk about new topics.  
- Range: `-2.0 – 2.0`  
- Higher values → increase diversity by discouraging sticking to the same topics.  


In [30]:
def set_open_params(
    model="gpt-3.5-turbo",
    temperature=0.7,
    max_tokens=256,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
):
    """ set openai parameters"""

    openai_params = {}    

    openai_params['model'] = model
    openai_params['temperature'] = temperature
    openai_params['max_tokens'] = max_tokens
    openai_params['top_p'] = top_p
    openai_params['frequency_penalty'] = frequency_penalty
    openai_params['presence_penalty'] = presence_penalty
    return openai_params

def get_completion(params, messages):
    """ GET completion from openai api"""

    response = openai.chat.completions.create(
        model = params['model'],
        messages = messages,
        temperature = params['temperature'],
        max_tokens = params['max_tokens'],
        top_p = params['top_p'],
        frequency_penalty = params['frequency_penalty'],
        presence_penalty = params['presence_penalty'],
    )
    return response

## 2. Advanced Prompting Techniques

Objectives:

- Cover more advanced techniques for prompting: few-shot, chain-of-thoughts,...

# Prompting Techniques in LLMs

## 1. Zero-Shot Prompting
- **Definition:** Asking the model to perform a task without giving any prior examples.  
- **When to Use:** Simple tasks where the model already has enough knowledge.  
- **Example:**  
  > "Translate the following English sentence to French: 'How are you today?'"

---

## 2. Few-Shot Prompting
- **Definition:** Providing the model with a few examples (2–5 typically) before asking it to complete a similar task.  
- **When to Use:** Tasks where context/examples help the model understand the expected style or format.  
- **Example:**  
English: "Hello" → French: "Bonjour"

English: "Good night" → French: "Bonne nuit"

English: "Thank you" → French: ?

---

## 3. Chain-of-Thought (CoT) Prompting
- **Definition:** Instructing the model to generate intermediate reasoning steps before giving the final answer.  
- **When to Use:** Complex reasoning problems like math, logic puzzles, or multi-step reasoning.  
- **Example:**  
> "If a train travels 60 km in 1.5 hours, what is its average speed?  
Let's reason step by step."  
- Model Output:  
  "60 ÷ 1.5 = 40. So the average speed is 40 km/h."

---

## 4. Zero-Shot Chain-of-Thought (Zero-Shot CoT)
- **Definition:** Similar to zero-shot, but you explicitly add a reasoning instruction like *“Let’s think step by step”* without giving examples.  
- **When to Use:** To encourage reasoning even when no examples are given.  
- **Example:**  
> "A shopkeeper buys a pen for $10 and sells it for $15. What is the profit?  
Let's think step by step."  
- Model Output:  
  "Cost price = $10, Selling price = $15, Profit = 15 – 10 = $5."


### 2.2 Few-shot prompts

In [25]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: The answer is False.

The odd numbers in this group add up to an even number: 17,  10, 19, 4, 8, 12, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 16,  11, 14, 4, 8, 13, 24.
A: The answer is True.

The odd numbers in this group add up to an even number: 17,  9, 10, 12, 13, 4, 2.
A: The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 
A:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

The answer is False.

### 2.3 Chain-of-Thought (CoT) Prompting
Introduced in Wei et al. (2022), chain-of-thought (CoT) prompting enables complex reasoning capabilities through intermediate reasoning steps. You can combine it with few-shot prompting to get better results on more complex tasks that require reasoning before responding.

In [26]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: Adding all the odd numbers (9, 15, 1) gives 25. The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 
A:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

Adding all the odd numbers (15, 5, 13, 7, 1) gives 41. The answer is False.

In [31]:
prompt = """The odd numbers in this group add up to an even number: 4, 8, 9, 15, 12, 2, 1.
A: Adding all the odd numbers (9, 15, 1) gives 25. The answer is False.

The odd numbers in this group add up to an even number: 17,  10, 19, 4, 8, 12, 24.
A: Adding all the odd numbers (17, 19) gives 36. The answer is True.

The odd numbers in this group add up to an even number: 16,  11, 14, 4, 8, 13, 24.
A: Adding all the odd numbers (11, 13) gives 24. The answer is True.

The odd numbers in this group add up to an even number: 17,  9, 10, 12, 13, 4, 2.
A: Adding all the odd numbers (17, 9, 13) gives 39. The answer is False.

The odd numbers in this group add up to an even number: 15, 32, 5, 13, 82, 7, 1. 
A:"""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

Adding all the odd numbers (15, 5, 13, 7, 1) gives 41. The answer is False.

### 2.4 Zero-shot CoT

In [28]:
prompt = """I went to the market and bought 10 apples. I gave 2 apples to the neighbor and 2 to the repairman. I then went and bought 5 more apples and ate 1. How many apples did I remain with?

Let's think step by step."""

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

response = get_completion(params, messages)
IPython.display.Markdown(response.choices[0].message.content)

Step 1: Bought 10 apples.
Step 2: Gave 2 apples to the neighbor and 2 apples to the repairman.
Remaining apples: 10 - 2 - 2 = 6 apples.
Step 3: Bought 5 more apples.
Total apples now: 6 + 5 = 11 apples.
Step 4: Ate 1 apple.
Remaining apples: 11 - 1 = 10 apples.

Final answer: You remained with 10 apples.

---